In [3]:
import os
import pickle
import warnings
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")

import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1

d:\Dissertation Mohit\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df = pd.read_csv('D:\\Dissertation Mohit\\data\\processed\\train\\forex_data_processed_train.csv', parse_dates=['time'], index_col='time')

In [9]:
df.head()

,price,spread
time,,
2024-03-31 21:00:00,1.07930,0.00088
2024-03-31 21:15:00,1.07947,0.00064
2024-03-31 21:30:00,1.07922,0.00052
2024-03-31 21:45:00,1.07934,0.00075
2024-03-31 22:00:00,1.07954,0.00018


In [10]:
df.drop(columns='spread', inplace=True)

In [12]:
df['return'] = np.log(df['price'] / df['price'].shift(1))
df['dir'] = (df['return'] > 0).astype(int)

In [13]:
def set_seeds(seed = 100):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
def cw(df):
    c0, c1 = np.bincount(df["dir"])
    w0 = (1/c0) * (len(df)) / 2
    w1 = (1/c1) * (len(df)) / 2
    return {0:w0, 1:w1}

In [14]:
def make_sequences(df, features=("return",), target="dir", lookback=50):
    data = df[list(features) + [target]].dropna().copy()

    X, y = [], []
    values = data[list(features)].values
    target_values = data[target].values

    for i in range(lookback, len(data)):
        X.append(values[i - lookback:i])   # shape: (lookback, n_features)
        y.append(target_values[i])

    return np.array(X), np.array(y)


def create_model(hl=2, hu=64, dropout=False, rate=0.3, regularize=False,
                 reg=l1(0.0005), optimizer=None, input_shape=None):
    if not regularize:
        reg = None
    if optimizer is None:
        optimizer = Adam(learning_rate=0.0001)

    model = Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))

    for i in range(hl):
        model.add(
            tf.keras.layers.LSTM(
                hu,
                return_sequences=(i < hl - 1),
                kernel_regularizer=reg
            )
        )
        if dropout:
            model.add(Dropout(rate, seed=100))

    model.add(Dense(1, activation="sigmoid"))
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model


# Build LSTM-ready data from existing df
lookback = 50
X, y = make_sequences(df, features=("return",), target="dir", lookback=lookback)

# Time-based split (no shuffle for time series)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train
set_seeds(100)
model = create_model(
    hl=2,
    hu=64,
    dropout=True,
    rate=0.2,
    regularize=False,
    input_shape=(X_train.shape[1], X_train.shape[2])
)

class_weight = cw(pd.DataFrame({"dir": y_train}))
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=256,
    validation_data=(X_test, y_test),
    class_weight=class_weight,
    verbose=1
)

Epoch 1/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 15s 100ms/step - accuracy: 0.4961 - loss: 0.6932 - val_accuracy: 0.4919 - val_loss: 0.6932
Epoch 2/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 12s 93ms/step - accuracy: 0.4953 - loss: 0.6932 - val_accuracy: 0.4919 - val_loss: 0.6932
Epoch 3/10
 46/130 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - accuracy: 0.5027 - loss: 0.6932

KeyboardInterrupt: 